In [2]:
import sys
import os
import warnings
import json
import pickle
import joblib
from datetime import datetime
import glob

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Modeling
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import learning_curve

# Suppress warnings
warnings.filterwarnings('ignore')

print(f"✅ Environment ready!")

✅ Environment ready!


In [3]:
# Setup paths and evaluation configuration
CONFIG = {
    'data': {
        'train_path': '../data/splits/splits/X_train.csv',
        'val_path': '../data/splits/splits/X_val.csv',
        'test_path': '../data/splits/splits/X_test.csv',
        'y_train_low': '../data/splits/splits/y_train_low.csv',
        'y_train_mid': '../data/splits/splits/y_train_mid.csv',
        'y_val_low': '../data/splits/splits/y_val_low.csv',
        'y_val_mid': '../data/splits/splits/y_val_mid.csv',
        'y_test_low': '../data/splits/splits/y_test_low.csv',
        'y_test_mid': '../data/splits/splits/y_test_mid.csv'
    },
    'models': {
        'checkpoint_dir': '../models/checkpoints/',
        'tuned_dir': '../models/tuned/',
        'final_dir': '../models/final/',
        'mlflow_dir': '../models/mlflow/'
    },
    'evaluation': {
        'random_state': 42,
        'test_size': 0.2
    }
}

# Create directories
os.makedirs(CONFIG['models']['final_dir'], exist_ok=True)
os.makedirs('../reports/figures/evaluation/', exist_ok=True)

print("✅ Configuration loaded!")

✅ Configuration loaded!


In [5]:
# Load validation data and all tuned models
def load_tuned_models(tuned_dir):
   
    print("\n📂 Loading tuned models...")
    
    model_files = glob.glob(f"{tuned_dir}/tuned_*.pkl")
    param_files = glob.glob(f"{tuned_dir}/params_*.json")
    
    if not model_files:
        print("❌ No tuned models found! Please run 'hyperparameter_tuning.ipynb' first.")
        return None
    
    models = {
        'low': {},
        'mid': {}
    }
    params = {
        'low': {},
        'mid': {}
    }
    
    for model_file in model_files:
        # Extract model info from filename
        filename = os.path.basename(model_file)
        parts = filename.replace('tuned_', '').replace('.pkl', '').split('_')
        
        # Handle model names with underscores (e.g., Random_Forest)
        if len(parts) == 3:
            model_name = parts[0]
            target_group = parts[1]
        else:
            # For models with underscores in name
            model_name = '_'.join(parts[:-1])
            target_group = parts[-1]
        
        try:
            model = joblib.load(model_file)
            models[target_group][model_name] = model
            print(f"  ✅ Loaded {model_name} ({target_group})")
        except Exception as e:
            print(f"  ❌ Failed to load {model_name}: {str(e)}")
    
    # Load parameters
    for param_file in param_files:
        filename = os.path.basename(param_file)
        parts = filename.replace('params_', '').replace('.json', '').split('_')
        
        if len(parts) == 3:
            model_name = parts[0]
            target_group = parts[1]
        else:
            model_name = '_'.join(parts[:-1])
            target_group = parts[-1]
        
        try:
            with open(param_file, 'r') as f:
                param_data = json.load(f)
            params[target_group][model_name] = param_data
        except Exception as e:
            print(f"  ⚠️ Could not load params for {model_name}: {str(e)}")
    
    return {'models': models, 'params': params}

# Load models
tuned_data = load_tuned_models(CONFIG['models']['tuned_dir'])

if tuned_data is None:
    print("❌ No models to evaluate. Please run Phase 2 first.")
    raise SystemExit

models = tuned_data['models']
params = tuned_data['params']

print(f"\n✅ Loaded models:")
print(f"  Low Income: {len(models['low'])} models")
print(f"  Middle Income: {len(models['mid'])} models")


📂 Loading tuned models...
❌ No tuned models found! Please run 'hyperparameter_tuning.ipynb' first.
❌ No models to evaluate. Please run Phase 2 first.


SystemExit: 